In [14]:
import requests
import pandas as pd
import time


def get_all_from_api(base_url, params=None, page_size=50):
    if params is None:
        params = {}

    all_data = []
    page = 1

    while True:
        params.update({'page': page, 'pageSize': page_size})

        try:
            response = requests.get(base_url, params=params, timeout=10)
            response.raise_for_status()
            data = response.json()

            if not data:
                break

            all_data.extend(data)
            print(f"Получена страница {page}: {len(data)} записей")

            if len(data) < page_size:
                break

            page += 1
            time.sleep(0.1)

        except requests.exceptions.RequestException as e:
            print(f"Ошибка при получении страницы {page}: {e}")
            break

    return pd.DataFrame(all_data) if all_data else pd.DataFrame()



books_df = get_all_from_api("https://www.anapioficeandfire.com/api/books")
print(f"\nВсего получено книг: {len(books_df)}")
if not books_df.empty:
    print(books_df[['name', 'isbn', 'numberOfPages']].head())


houses_df = get_all_from_api("https://www.anapioficeandfire.com/api/houses")
print(f"\nВсего получено домов: {len(houses_df)}")


houses_with_words_df = houses_df[houses_df['words'].notna() & (houses_df['words'].str.strip() != "")]
print(f"Всего домов с девизом: {len(houses_with_words_df)}")

print(houses_with_words_df[['name', 'words']].head(10))


print(f"\nВсего домов в API: {len(houses_df)}")
print(f"Домов с девизом: {len(houses_with_words_df)}")
print(f"Домов без девиза: {len(houses_df) - len(houses_with_words_df)}")


Получена страница 1: 12 записей

Всего получено книг: 12
                name            isbn  numberOfPages
0  A Game of Thrones  978-0553103540            694
1   A Clash of Kings  978-0553108033            768
2  A Storm of Swords  978-0553106633            992
3   The Hedge Knight  978-0976401100            164
4  A Feast for Crows  978-0553801507            784
Получена страница 1: 50 записей
Получена страница 2: 50 записей
Получена страница 3: 50 записей
Получена страница 4: 50 записей
Получена страница 5: 50 записей
Получена страница 6: 50 записей
Получена страница 7: 50 записей
Получена страница 8: 50 записей
Получена страница 9: 44 записей

Всего получено домов: 444
Всего домов с девизом: 68
                              name                  words
1      House Allyrion of Godsgrace        No Foe May Pass
3                    House Ambrose          Never Resting
6         House Arryn of the Eyrie       As High as Honor
7         House Ashford of Ashford  Our Sun Shines Bright


In [17]:
# Параметры подключения к БД (замените на реальные)
DB_CONFIG = {
    'host': 'hh-pgsql-public.ebi.ac.uk',
    'database': 'pfmegrnargs',
    'user': 'reader',
    'password': 'NWDMCE5xdipIjRrp',
    'port': '5432'
}

def connect_to_db():
    try:
        conn = psycopg2.connect(**DB_CONFIG)
        print("Успешное подключение к БД")
        return conn
    except Exception as e:
        print(f"Ошибка подключения к БД: {e}")
        return None

def get_10_rows(conn):
    try:
        cursor = conn.cursor()
        cursor.execute("SELECT * FROM rnc_database LIMIT 10")
        rows = cursor.fetchall()

        column_names = [desc[0] for desc in cursor.description]

        cursor.close()
        return rows, column_names
    except Exception as e:
        print(f"Ошибка при выполнении запроса: {e}")
        return [], []

def get_specific_columns(conn):
    try:
        cursor = conn.cursor()
        query = """
        SELECT display_name, num_sequences, num_organisms, url
        FROM rnc_database
        LIMIT 10
        """
        cursor.execute(query)
        rows = cursor.fetchall()

        column_names = ['display_name', 'num_sequences', 'num_organisms', 'url']

        cursor.close()
        return rows, column_names
    except Exception as e:
        print(f"Ошибка при выполнении запроса: {e}")
        return [], []


connection = connect_to_db()

if connection:
    print("\n1. Получаем 10 строк из таблицы rnc_database...")
    rows, columns = get_10_rows(connection)

    if rows:
        full_data_df = pd.DataFrame(rows, columns=columns)
        print("Данные сохранены в DataFrame:")
        print(full_data_df.head())

    print("\n2. Получаем конкретные столбцы из таблицы rnc_database...")
    specific_rows, specific_columns = get_specific_columns(connection)

    if specific_rows:
        specific_data_df = pd.DataFrame(specific_rows, columns=specific_columns)
        print("Данные с конкретными столбцами сохранены в DataFrame:")
        print(specific_data_df)

    connection.close()
    print("\nСоединение с БД прервано")
else:
    print("Не удалось подключиться к БД")

Успешное подключение к БД

1. Получаем 10 строк из таблицы rnc_database...
Данные сохранены в DataFrame:
   id  timestamp userstamp    descr  current_release full_descr alive  \
0  21 2017-05-02    RNACEN  NONCODE              146    NONCODE     Y   
1   5 2017-05-17    RNACEN     VEGA               98       VEGA     N   
2  26 2017-05-01    RNACEN  GENCODE              450    GENCODE     N   
3   1 2017-05-01    RNACEN      ENA              968        ENA     Y   
4  14 2017-05-01    RNACEN     TAIR              982       TAIR     Y   

  for_release display_name project_id  avg_length  min_length  max_length  \
0                  NONCODE                 1130.0       201.0    244296.0   
1                     VEGA  PRJEB4568         NaN         NaN         NaN   
2                  GENCODE                  889.0        32.0    205012.0   
3                      ENA                  412.0        10.0    900074.0   
4                     TAIR   PRJ_TAIR       384.0        19.0      6227